# Análisis de Propagación Forward de Spikes
## Caracterización de Trayectorias y Branching Ratio (σ)

**Objetivo**: Medir probabilidad de transmisión y branching ratio empíricamente.  
**Enfoque**: Análisis E→E (vecinos a orden 1)

---

In [ ]:
# Setup
import os
import sys
from pathlib import Path

if Path.cwd().name == 'two_populations':
    os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt
from brian2 import *
from datetime import datetime
from collections import defaultdict

from src.two_populations.model import IzhikevichNetwork
from src.two_populations.metrics import analyze_simulation_results, print_network_statistics_table
from src.two_populations.plots.basic_plots import plot_raster_results
from src.two_populations.plots.dashboard_plots import plot_population_dashboard, plot_connectivity_dashboard
from src.two_populations.helpers.logger import setup_logger
from src.two_populations.helpers.validator import (
    add_validation_to_analysis, 
    plot_population_validation_dashboard, 
    print_validation_summary
)
from src.two_populations.metrics import analyze_simulation_results
from src.two_populations.plots.basic_plots import plot_raster_results, plot_voltage_traces

logger = setup_logger(
    experiment_name="spike_propagation",
    console_level="INFO",
    file_level="DEBUG",
    log_to_file=False
)

logger.info(f"Working directory: {Path.cwd()}")

## 1. Configuración de Simulación

In [ ]:
# Parámetros de simulación
Ni = 200
Ne = 800
time = 2500

SIM_CONFIG = {
    'dt_ms': 0.1,
    'T_ms': time,
    'warmup_ms': 500
}

# Parámetros base (estímulo constante)
BASE_PARAMS = {
    'Ne': Ne, 'Ni': Ni,
    'noise_exc': 0.884, 
    'noise_inh': 0.60,
    'p_intra': 0.1, 
    'delay': 0.0,
    'rate_hz': 8.0,
    'stim_start_ms': None, 
    'stim_duration_ms': time,  # Estímulo constante
    'stim_base': 1.0,          # Sin modulación
    'stim_elevated': None
}

logger.info(f"Configuración: Ne={Ne}, Ni={Ni}, T={time}ms (estímulo constante)")

In [ ]:
# =============================================================================
# ANALYSIS WITH VALIDATION
# =============================================================================

def results_summary(results, network):
    # Analyze connectivity
    results_dict = {
        'baseline': analyze_simulation_results(
        results['A']['spike_monitor'], 
        None, 
        1000, "Baseline", 
        warmup=SIM_CONFIG['warmup_ms'],
        state_monitors={'A': network.monitors['A']},
        signal_mode = 'lfp',
        T_total=SIM_CONFIG['T_ms'] # firing_rate
        )
    }
    

    # Add validation metrics
    validation_results = add_validation_to_analysis(results_dict)

    # Network statistics table
    print_network_statistics_table(
        results, network, 
        N_exc=Ne, N_inh=Ni, 
        T_total=SIM_CONFIG['T_ms'], 
        warmup=500 #SIM_CONFIG['warmup_ms']
    )
    
    fig_conn = plot_raster_results(results)
    fig = plot_voltage_traces(results_dict, results, (500,1500), 3, 3, True, 3)
    
    # Dashboards
    fig_val = plot_population_validation_dashboard(validation_results)
    print_validation_summary(validation_results)

    fig_pop = plot_population_dashboard(results_dict)
    plt.show()

## 2. Simulación Baseline (K=0)

In [ ]:
# K=0: Solo input externo, sin acoplamiento recurrente
start_scope()
k_factor = 0.0

network_baseline = IzhikevichNetwork(
    dt_val=SIM_CONFIG['dt_ms'],
    T_total=SIM_CONFIG['T_ms'],
    fixed_seed=100,
    variable_seed=200,
    trial=0
)

params_baseline = {**BASE_PARAMS, 'k_exc': k_factor, 'k_inh': k_factor*3.9}
pop_A_baseline = network_baseline.create_population2(name='A', **params_baseline)

network_baseline.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
results_baseline = network_baseline.run_simulation()

logger.success(f"Baseline K=0: {len(results_baseline['A']['spike_times'])} spikes")

results_summary(results_baseline, network_baseline)

In [ ]:
sum(network_baseline.populations['A']['syn_intra'].w)

## 3. Simulación con Red Acoplada (K>0)

In [ ]:
# K=0: Solo input externo, sin acoplamiento recurrente
start_scope()
k_factor = 2.5

network_coupled = IzhikevichNetwork(
    dt_val=SIM_CONFIG['dt_ms'],
    T_total=SIM_CONFIG['T_ms'],
    fixed_seed=100,
    variable_seed=200,
    trial=0
)

params_coupled = {**BASE_PARAMS, 'k_exc': k_factor, 'k_inh': k_factor*3.9}
pop_A_coupled = network_coupled.create_population2(name='A', **params_coupled)

network_coupled.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
results_coupled = network_coupled.run_simulation()

logger.success(f"Baseline K=0: {len(results_coupled['A']['spike_times'])} spikes")

results_summary(results_coupled, network_coupled)

In [ ]:
sum(network_coupled.populations['A']['syn_intra'].w)

## 4. Clase de Análisis Forward

In [ ]:
class ForwardPropagationAnalyzer:
    """
    Analiza propagación forward: cuando neurona i dispara, ¿cuántos vecinos j responden?
    
    Métricas:
        - P_transmission: probabilidad de activación por spike
        - σ (sigma): branching ratio = <n_activados>
    """
    
    def __init__(self, window_ms=5.0):
        self.window = window_ms
    
    def extract_connectivity_E2E(self, synapses_intra, Ne, min_weight):
        """Extrae conectividad estructural E→E"""
        neighbors = defaultdict(list)
        weights = {}
        
        pre_indices = np.array(synapses_intra.i)
        post_indices = np.array(synapses_intra.j)
        syn_weights = np.array(synapses_intra.w)
        
        # Stats pre-filtro
        total_synapses = len(pre_indices)
        E2E_mask = (pre_indices < Ne) & (post_indices < Ne) & (syn_weights >= 0.0)
        n_E2E = np.sum(E2E_mask)
        
        # Filtro por peso
        mask = E2E_mask & (syn_weights >= min_weight)
        n_filtered = np.sum(mask)
        
        logger.info(f"  Total synapses: {total_synapses}")
        logger.info(f"  E→E (w>0): {n_E2E}")
        logger.info(f"  E→E (w>{min_weight}): {n_filtered} ({100*n_filtered/n_E2E:.1f}%)")
        if n_filtered > 0:
            logger.info(f"  Weight range: [{syn_weights[mask].min():.3f}, {syn_weights[mask].max():.3f}]")
        else:
            logger.warning(f"  No connections pass min_weight threshold!")
        
        for pre, post, w in zip(pre_indices[mask], post_indices[mask], syn_weights[mask]):
            neighbors[int(pre)].append(int(post))
            weights[(int(pre), int(post))] = float(w)
        
        # Out-degree stats
        degrees = [len(v) for v in neighbors.values()]
        if degrees:
            logger.info(f"  Out-degree: mean={np.mean(degrees):.1f}, median={np.median(degrees):.0f}, max={np.max(degrees)}")
        
        return dict(neighbors), weights
        
    def organize_spike_times(self, spike_times_arr, spike_indices_arr):
        """Organiza spikes por neurona"""
        spike_dict = defaultdict(list)
        
        for t, idx in zip(spike_times_arr, spike_indices_arr):
            spike_dict[int(idx)].append(float(t))
        
        spike_dict = {k: np.sort(v) for k, v in spike_dict.items()}
        return dict(spike_dict)
    
    def count_responses_single_spike(self, pre_spike_time, post_neuron_spikes):
        """Verifica si post respondió en ventana temporal"""
        if len(post_neuron_spikes) == 0:
            return False
        
        responses = post_neuron_spikes[(post_neuron_spikes > pre_spike_time) &  #TODO DEFINIRMOS > O >=??
                                    (post_neuron_spikes < pre_spike_time + self.window)]
        return len(responses) > 0
    
    def analyze_single_spikes(self, spike_dict, neighbors, min_spikes=5):
        """Análisis principal con logging"""
        logger.info(f"  Neurons with spikes: {len(spike_dict)}")
        logger.info(f"  Neurons with neighbors: {len(neighbors)}")
        logger.info(f"  Min spikes threshold: {min_spikes}")
        
        ratios_per_spike = []
        activated_counts = []
        per_neuron_stats = {}
        
        total_spikes_analyzed = 0
        successful_transmissions = 0
        neurons_excluded_min_spikes = 0
        
        for pre_idx in neighbors.keys():
            if pre_idx not in spike_dict:
                continue
            
            pre_spikes = spike_dict[pre_idx]
            
            if len(pre_spikes) < min_spikes:
                neurons_excluded_min_spikes += 1
                continue
            
            post_neighbors = neighbors[pre_idx]
            n_neighbors = len(post_neighbors)
            
            if n_neighbors == 0:
                continue
            
            neuron_ratios = []
            neuron_activated = []
            
            for spike_time in pre_spikes:
                n_activated = 0
                
                for post_idx in post_neighbors:
                    if post_idx not in spike_dict:
                        continue
                    
                    post_spikes = spike_dict[post_idx]
                    
                    if self.count_responses_single_spike(spike_time, post_spikes):
                        n_activated += 1
                
                ratio = n_activated / n_neighbors
                
                ratios_per_spike.append(ratio)
                activated_counts.append(n_activated)
                neuron_ratios.append(ratio)
                neuron_activated.append(n_activated)
                
                total_spikes_analyzed += 1
                if n_activated > 0:
                    successful_transmissions += 1
            
            per_neuron_stats[pre_idx] = {
                'n_spikes': len(pre_spikes),
                'n_neighbors': n_neighbors,
                'mean_ratio': np.mean(neuron_ratios),
                'std_ratio': np.std(neuron_ratios),
                'mean_activated': np.mean(neuron_activated),
                'success_rate': np.sum(np.array(neuron_activated) > 0) / len(neuron_activated)
            }
            
        logger.info(f"  Neurons excluded (min_spikes): {neurons_excluded_min_spikes}")
        logger.info(f"  Neurons analyzed: {len(per_neuron_stats)}")
        logger.info(f"  Total spikes analyzed: {total_spikes_analyzed}")
        
        ratios_per_spike = np.array(ratios_per_spike)
        activated_counts = np.array(activated_counts)
        
        results = {
            'P_transmission_global': np.mean(ratios_per_spike) if len(ratios_per_spike) > 0 else 0.0,
            'sigma': np.mean(activated_counts) if len(activated_counts) > 0 else 0.0,
            'ratio_distribution': ratios_per_spike,
            'activated_counts': activated_counts,
            'stats': {
                'n_total_spikes': total_spikes_analyzed,
                'n_successful': successful_transmissions,
                'n_failed': total_spikes_analyzed - successful_transmissions,
                'success_rate': successful_transmissions / total_spikes_analyzed if total_spikes_analyzed > 0 else 0.0,
                'n_neurons_analyzed': len(per_neuron_stats),
                'std_ratio': np.std(ratios_per_spike) if len(ratios_per_spike) > 0 else 0.0,
                'std_sigma': np.std(activated_counts) if len(activated_counts) > 0 else 0.0
            },
            'per_neuron': per_neuron_stats,
            'window_ms': self.window
        }
        
        return results

logger.info("ForwardPropagationAnalyzer cargado")

## 5. Análisis de Propagación: Baseline vs Coupled

In [ ]:
analyzer = ForwardPropagationAnalyzer(window_ms=5.0)

# === BASELINE (K=0) ===
logger.info("Analizando baseline (K=0)...")
spike_dict_baseline = analyzer.organize_spike_times(
    results_baseline['A']['spike_times'], 
    results_baseline['A']['spike_indices']
)
neighbors_baseline, _ = analyzer.extract_connectivity_E2E(
    network_baseline.populations['A']['syn_intra'], Ne, 0.0
)
prop_baseline = analyzer.analyze_single_spikes(
    spike_dict_baseline, neighbors_baseline, min_spikes=10
)

# === COUPLED (K>0) ===
logger.info("Analizando coupled (K>0)...")
spike_dict_coupled = analyzer.organize_spike_times(
    results_coupled['A']['spike_times'], 
    results_coupled['A']['spike_indices']
)
neighbors_coupled, _ = analyzer.extract_connectivity_E2E(
    network_coupled.populations['A']['syn_intra'], Ne, 0.0
)
prop_coupled = analyzer.analyze_single_spikes(
    spike_dict_coupled, neighbors_coupled, min_spikes=10
)

logger.success("Análisis completado para ambas condiciones")

## 6. Resultados Comparativos

In [ ]:
print("="*70)
print("COMPARACIÓN: BASELINE (K=0) vs COUPLED (K>0)")
print("="*70)

print(f"\n{'Métrica':<25} {'K=0 (baseline)':<20} {'K={k_factor} (coupled)':<20} {'Δ':<15}")
print("-"*70)

# P_transmission
P_base = prop_baseline['P_transmission_global']
P_coup = prop_coupled['P_transmission_global']
print(f"{'P_transmission':<25} {P_base:<20.4f} {P_coup:<20.4f} {P_coup-P_base:+.4f}")

# σ (branching)
sigma_base = prop_baseline['sigma']
sigma_coup = prop_coupled['sigma']
print(f"{'σ (branching)':<25} {sigma_base:<20.4f} {sigma_coup:<20.4f} {sigma_coup-sigma_base:+.4f}")

# Success rate
sr_base = prop_baseline['stats']['success_rate']
sr_coup = prop_coupled['stats']['success_rate']
print(f"{'Success rate':<25} {sr_base:<20.4f} {sr_coup:<20.4f} {sr_coup-sr_base:+.4f}")

# N spikes
n_base = prop_baseline['stats']['n_total_spikes']
n_coup = prop_coupled['stats']['n_total_spikes']
print(f"{'N spikes analyzed':<25} {n_base:<20} {n_coup:<20} {n_coup-n_base:+}")

print("\n" + "="*70)
print("INTERPRETACIÓN")
print("="*70)

if sigma_base > 0.05:
    print(f"\n⚠️  BASELINE: σ={sigma_base:.3f} > 0 → Correlaciones espurias del input")
else:
    print(f"\n✓  BASELINE: σ≈0 → Sin propagación estructural (como esperado)")

if sigma_coup < 1.0:
    print(f"✓  COUPLED: σ={sigma_coup:.3f} < 1 → Red subcrítica")
elif sigma_coup > 1.0:
    print(f"⚠️  COUPLED: σ={sigma_coup:.3f} > 1 → Red supercrítica")
else:
    print(f"✓  COUPLED: σ≈1 → Red crítica (óptimo)")

fold_change = sigma_coup / sigma_base if sigma_base > 0 else float('inf')
print(f"\n📈 Factor de incremento σ: {fold_change:.1f}x")

In [ ]:
# Después de prop_baseline y prop_coupled
fig_diag, axes_diag = plt.subplots(2, 3, figsize=(15, 8))

for idx, (prop, neighbors, title) in enumerate([
    (prop_baseline, neighbors_baseline, 'Baseline'),
    (prop_coupled, neighbors_coupled, 'Coupled')
]):
    # Out-degrees
    ax = axes_diag[idx, 0]
    degrees = [prop['per_neuron'][nid]['n_neighbors'] 
               for nid in prop['per_neuron'].keys()]
    ax.hist(degrees, bins=20, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Out-degree')
    ax.set_ylabel('Count')
    ax.set_title(f'{title}: Out-degree dist\n(mean={np.mean(degrees):.1f})')
    ax.grid(alpha=0.3)
    
    # Mean activated per neuron
    ax = axes_diag[idx, 1]
    mean_act = [prop['per_neuron'][nid]['mean_activated'] 
                for nid in prop['per_neuron'].keys()]
    ax.hist(mean_act, bins=20, alpha=0.7, edgecolor='black')
    ax.axvline(prop['sigma'], color='red', linestyle='--', 
               label=f'Global σ={prop["sigma"]:.2f}')
    ax.set_xlabel('Mean activated per neuron')
    ax.set_ylabel('Count')
    ax.set_title(f'{title}: Heterogeneity')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # Success rate per neuron
    ax = axes_diag[idx, 2]
    sr = [prop['per_neuron'][nid]['success_rate'] 
          for nid in prop['per_neuron'].keys()]
    ax.hist(sr, bins=20, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Success rate')
    ax.set_ylabel('Count')
    ax.set_title(f'{title}: P(spike activates ≥1)')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualizaciones Comparativas

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# === COLUMNA 1: RASTER PLOTS ===
for idx, (res, title, color) in enumerate([
    (results_baseline, 'Baseline (K=0)', 'gray'),
    (results_coupled, f'Coupled (K={k_factor})', 'black')
]):
    ax = axes[idx, 0]
    ax.scatter(res['A']['spike_times'], res['A']['spike_indices'], 
               s=0.5, c=color, alpha=0.6)
    ax.axhline(Ne, color='red', linestyle='--', alpha=0.5, linewidth=1)
    ax.set_ylabel('Neuron index', fontsize=10)
    ax.set_title(title, fontsize=11)
    if idx == 1:
        ax.set_xlabel('Time (ms)', fontsize=10)

# === COLUMNA 2: DISTRIBUCIÓN DE RATIOS ===
for idx, (prop, title, color) in enumerate([
    (prop_baseline, 'Baseline', 'gray'),
    (prop_coupled, 'Coupled', 'steelblue')
]):
    ax = axes[idx, 1]
    ratios = prop['ratio_distribution']
    ax.hist(ratios, bins=30, density=True, alpha=0.7, 
            edgecolor='black', color=color)
    ax.axvline(np.mean(ratios), color='red', linestyle='--', 
               linewidth=2, label=f'Mean={np.mean(ratios):.3f}')
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'P_transmission ({title})', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    if idx == 1:
        ax.set_xlabel('Ratio', fontsize=10)

# === COLUMNA 3: BRANCHING RATIO ===
for idx, (prop, title, color) in enumerate([
    (prop_baseline, 'Baseline', 'gray'),
    (prop_coupled, 'Coupled', 'coral')
]):
    ax = axes[idx, 2]
    activated = prop['activated_counts']
    
    if len(activated) == 0:  # <-- Agregar esto
        ax.text(0.5, 0.5, 'No data\n(w>0.5 filter)', 
                ha='center', va='center', transform=ax.transAxes)
        ax.set_ylabel('Density', fontsize=10)
        ax.set_title(f'σ Distribution ({title})', fontsize=11)
        continue
    
    bins = np.arange(-0.5, min(np.max(activated)+1.5, 20), 1)
    ax.hist(activated, bins=bins, density=True, alpha=0.7, 
            edgecolor='black', color=color)
    ax.axvline(np.mean(activated), color='red', linestyle='--', 
               linewidth=2, label=f'σ={np.mean(activated):.3f}')
    ax.axvline(1.0, color='green', linestyle=':', linewidth=2, 
               alpha=0.7, label='Critical')
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'σ Distribution ({title})', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    if idx == 1:
        ax.set_xlabel('N activated', fontsize=10)

plt.tight_layout()
plt.show()

logger.success("Visualizaciones comparativas generadas")

In [ ]:
# Barrido de min_spikes
min_spikes_vals = np.arange(1,20,2)
results_sweep = []

for ms in min_spikes_vals:
    analyzer = ForwardPropagationAnalyzer(window_ms=5.0)
    
    # Baseline
    neigh_b, _ = analyzer.extract_connectivity_E2E(
        network_baseline.populations['A']['syn_intra'], Ne, 0.0
    )
    prop_b = analyzer.analyze_single_spikes(
        spike_dict_baseline, neigh_b, min_spikes=ms
    )
    
    # Coupled
    neigh_c, _ = analyzer.extract_connectivity_E2E(
        network_coupled.populations['A']['syn_intra'], Ne, 0.5
    )
    prop_c = analyzer.analyze_single_spikes(
        spike_dict_coupled, neigh_c, min_spikes=ms
    )
    
    results_sweep.append({
        'min_spikes': ms,
        'baseline': prop_b,
        'coupled': prop_c,
        'delta_sigma': prop_c['sigma'] - prop_b['sigma'],
        'n_neurons_b': prop_b['stats']['n_neurons_analyzed'],
        'n_neurons_c': prop_c['stats']['n_neurons_analyzed']
    })

# Plot comparativo
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# σ vs min_spikes
ax = axes[0, 0]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['baseline']['sigma'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['coupled']['sigma'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.axhline(1, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('min_spikes')
ax.set_ylabel('σ')
ax.legend()
ax.grid(alpha=0.3)

# Δσ vs min_spikes
ax = axes[0, 1]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['delta_sigma'] for r in results_sweep], 
        'o-', color='purple', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('Δσ (structural)')
ax.grid(alpha=0.3)

# N neurons analyzed (log scale)
ax = axes[1, 0]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['n_neurons_b'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['n_neurons_c'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('N neurons analyzed')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)

# P_transmission vs min_spikes
ax = axes[1, 1]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['baseline']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['coupled']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('P_transmission')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "="*90)
print("BARRIDO MIN_SPIKES")
print("="*90)
print(f"{'min_spikes':<12} {'σ_base':<10} {'σ_coupled':<12} {'Δσ':<10} {'N_base':<8} {'N_coupled':<10}")
print("-"*90)
for r in results_sweep:
    print(f"{r['min_spikes']:<12} {r['baseline']['sigma']:<10.3f} "
          f"{r['coupled']['sigma']:<12.3f} {r['delta_sigma']:<10.3f} "
          f"{r['n_neurons_b']:<8} {r['n_neurons_c']:<10}")

In [ ]:
# Barrido de window temporal
windows = np.arange(0.5, 15, 0.5)
results_sweep = []

for w in windows:
    analyzer = ForwardPropagationAnalyzer(window_ms=w)
    
    # Baseline
    neigh_b, _ = analyzer.extract_connectivity_E2E(
        network_baseline.populations['A']['syn_intra'], Ne, 0.0
    )
    prop_b = analyzer.analyze_single_spikes(
        spike_dict_baseline, neigh_b, min_spikes=10
    )
    
    # Coupled
    neigh_c, _ = analyzer.extract_connectivity_E2E(
        network_coupled.populations['A']['syn_intra'], Ne, 0.5
    )
    prop_c = analyzer.analyze_single_spikes(
        spike_dict_coupled, neigh_c, min_spikes=10
    )
    
    results_sweep.append({
        'window': w,
        'baseline': prop_b,
        'coupled': prop_c,
        'delta_sigma': prop_c['sigma'] - prop_b['sigma'],
        'n_neurons_b': prop_b['stats']['n_neurons_analyzed'],
        'n_neurons_c': prop_c['stats']['n_neurons_analyzed']
    })

# Plot comparativo
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# σ vs window
ax = axes[0, 0]
ax.plot([r['window'] for r in results_sweep], 
        [r['baseline']['sigma'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['window'] for r in results_sweep], 
        [r['coupled']['sigma'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.axhline(1, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('Window (ms)')
ax.set_ylabel('σ')
ax.legend()
ax.grid(alpha=0.3)

# Δσ vs window
ax = axes[0, 1]
ax.plot([r['window'] for r in results_sweep], 
        [r['delta_sigma'] for r in results_sweep], 
        'o-', color='purple', linewidth=2)
ax.set_xlabel('Window (ms)')
ax.set_ylabel('Δσ (structural)')
ax.grid(alpha=0.3)

# N neurons analyzed
ax = axes[1, 0]
ax.plot([r['window'] for r in results_sweep], 
        [r['n_neurons_b'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['window'] for r in results_sweep], 
        [r['n_neurons_c'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('Window (ms)')
ax.set_ylabel('N neurons analyzed')
ax.legend()
ax.grid(alpha=0.3)

# P_transmission vs window
ax = axes[1, 1]
ax.plot([r['window'] for r in results_sweep], 
        [r['baseline']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['window'] for r in results_sweep], 
        [r['coupled']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('Window (ms)')
ax.set_ylabel('P_transmission')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "="*90)
print("BARRIDO WINDOW TEMPORAL")
print("="*90)
print(f"{'window (ms)':<12} {'σ_base':<10} {'σ_coupled':<12} {'Δσ':<10} {'N_base':<8} {'N_coupled':<10}")
print("-"*90)
for r in results_sweep:
    print(f"{r['window']:<12.1f} {r['baseline']['sigma']:<10.3f} "
          f"{r['coupled']['sigma']:<12.3f} {r['delta_sigma']:<10.3f} "
          f"{r['n_neurons_b']:<8} {r['n_neurons_c']:<10}")

In [ ]:
# Barrido de min_spikes
min_spikes_vals = [1, 3, 5, 10, 15, 20]
results_sweep = []

for ms in min_spikes_vals:
    analyzer = ForwardPropagationAnalyzer(window_ms=5.0)
    
    # Baseline
    neigh_b, _ = analyzer.extract_connectivity_E2E(
        network_baseline.populations['A']['syn_intra'], Ne, 0.0
    )
    prop_b = analyzer.analyze_single_spikes(
        spike_dict_baseline, neigh_b, min_spikes=ms
    )
    
    # Coupled
    neigh_c, _ = analyzer.extract_connectivity_E2E(
        network_coupled.populations['A']['syn_intra'], Ne, 0.5
    )
    prop_c = analyzer.analyze_single_spikes(
        spike_dict_coupled, neigh_c, min_spikes=ms
    )
    
    results_sweep.append({
        'min_spikes': ms,
        'baseline': prop_b,
        'coupled': prop_c,
        'delta_sigma': prop_c['sigma'] - prop_b['sigma'],
        'n_neurons_b': prop_b['stats']['n_neurons_analyzed'],
        'n_neurons_c': prop_c['stats']['n_neurons_analyzed']
    })

# Plot comparativo
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# σ vs min_spikes
ax = axes[0, 0]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['baseline']['sigma'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['coupled']['sigma'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.axhline(1, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('min_spikes')
ax.set_ylabel('σ')
ax.legend()
ax.grid(alpha=0.3)

# Δσ vs min_spikes
ax = axes[0, 1]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['delta_sigma'] for r in results_sweep], 
        'o-', color='purple', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('Δσ (structural)')
ax.grid(alpha=0.3)

# N neurons analyzed (log scale)
ax = axes[1, 0]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['n_neurons_b'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['n_neurons_c'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('N neurons analyzed')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)

# P_transmission vs min_spikes
ax = axes[1, 1]
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['baseline']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_spikes'] for r in results_sweep], 
        [r['coupled']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_spikes')
ax.set_ylabel('P_transmission')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "="*90)
print("BARRIDO MIN_SPIKES")
print("="*90)
print(f"{'min_spikes':<12} {'σ_base':<10} {'σ_coupled':<12} {'Δσ':<10} {'N_base':<8} {'N_coupled':<10}")
print("-"*90)
for r in results_sweep:
    print(f"{r['min_spikes']:<12} {r['baseline']['sigma']:<10.3f} "
          f"{r['coupled']['sigma']:<12.3f} {r['delta_sigma']:<10.3f} "
          f"{r['n_neurons_b']:<8} {r['n_neurons_c']:<10}")

In [ ]:
max(network_coupled.populations['A']['syn_intra'].w)

In [ ]:
# Barrido de min_weight
min_weights = np.arange(0.0, 2.4, 0.2)
results_sweep = []

for mw in min_weights:
    analyzer = ForwardPropagationAnalyzer(window_ms=5.0)
    
    # Baseline
    neigh_b, _ = analyzer.extract_connectivity_E2E(
        network_baseline.populations['A']['syn_intra'], Ne, 0.0
    )
    prop_b = analyzer.analyze_single_spikes(
        spike_dict_baseline, neigh_b, min_spikes=10
    )
    
    # Coupled
    neigh_c, _ = analyzer.extract_connectivity_E2E(
        network_coupled.populations['A']['syn_intra'], Ne, mw
    )
    prop_c = analyzer.analyze_single_spikes(
        spike_dict_coupled, neigh_c, min_spikes=10
    )
    
    results_sweep.append({
        'min_weight': mw,
        'baseline': prop_b,
        'coupled': prop_c,
        'delta_sigma': prop_c['sigma'] - prop_b['sigma'],
        'n_neurons_b': prop_b['stats']['n_neurons_analyzed'],
        'n_neurons_c': prop_c['stats']['n_neurons_analyzed']
    })

# Plot comparativo
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# σ vs min_weight
ax = axes[0, 0]
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['baseline']['sigma'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['coupled']['sigma'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.axhline(1, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel('min_weight')
ax.set_ylabel('σ')
ax.legend()
ax.grid(alpha=0.3)

# Δσ vs min_weight
ax = axes[0, 1]
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['delta_sigma'] for r in results_sweep], 
        'o-', color='purple', linewidth=2)
ax.set_xlabel('min_weight')
ax.set_ylabel('Δσ (structural)')
ax.grid(alpha=0.3)

# N neurons analyzed
ax = axes[1, 0]
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['n_neurons_b'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['n_neurons_c'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_weight')
ax.set_ylabel('N neurons analyzed')
ax.legend()
ax.grid(alpha=0.3)

# P_transmission vs min_weight
ax = axes[1, 1]
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['baseline']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Baseline', linewidth=2)
ax.plot([r['min_weight'] for r in results_sweep], 
        [r['coupled']['P_transmission_global'] for r in results_sweep], 
        'o-', label='Coupled', linewidth=2)
ax.set_xlabel('min_weight')
ax.set_ylabel('P_transmission')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print("\n" + "="*90)
print("BARRIDO MIN_WEIGHT")
print("="*90)
print(f"{'min_weight':<12} {'σ_base':<10} {'σ_coupled':<12} {'Δσ':<10} {'N_base':<8} {'N_coupled':<10}")
print("-"*90)
for r in results_sweep:
    print(f"{r['min_weight']:<12.2f} {r['baseline']['sigma']:<10.3f} "
          f"{r['coupled']['sigma']:<12.3f} {r['delta_sigma']:<10.3f} "
          f"{r['n_neurons_b']:<8} {r['n_neurons_c']:<10}")

## 8. Análisis Multi-Trial (Opcional)

In [ ]:
# Ejecutar múltiples trials para ambas condiciones
n_trials = 5

def run_trial_comparison(trial_id, k_value):
    """Ejecuta un trial con K dado"""
    start_scope()
    
    net = IzhikevichNetwork(
        dt_val=SIM_CONFIG['dt_ms'],
        T_total=SIM_CONFIG['T_ms'],
        fixed_seed=100,
        variable_seed=200 + trial_id,
        trial=trial_id
    )
    
    params = {**BASE_PARAMS, 'k_exc': k_value, 'k_inh': k_value*3.9}
    net.create_population2(name='A', **params)
    net.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
    results = net.run_simulation()
    
    analyzer = ForwardPropagationAnalyzer(window_ms=5.0)
    neighbors, _ = analyzer.extract_connectivity_E2E(
        net.populations['A']['syn_intra'], Ne, 0.5
    )
    spike_dict = analyzer.organize_spike_times(
        results['A']['spike_times'], results['A']['spike_indices']
    )
    
    return analyzer.analyze_single_spikes(spike_dict, neighbors, min_spikes=5)

print(f"Ejecutando {n_trials} trials para baseline (K=0)...")
trials_baseline = [run_trial_comparison(i, 0.0) for i in range(n_trials)]

print(f"Ejecutando {n_trials} trials para coupled (K={k_factor})...")
trials_coupled = [run_trial_comparison(i, k_factor) for i in range(n_trials)]

# Agregar
for name, trials in [('Baseline', trials_baseline), ('Coupled', trials_coupled)]:
    sigmas = [t['sigma'] for t in trials]
    P_trans = [t['P_transmission_global'] for t in trials]
    
    print(f"\n{name}:")
    print(f"  σ = {np.mean(sigmas):.4f} ± {np.std(sigmas)/np.sqrt(n_trials):.4f}")
    print(f"  P = {np.mean(P_trans):.4f} ± {np.std(P_trans)/np.sqrt(n_trials):.4f}")

## 9. Barrido de Ventanas Temporales

In [ ]:
# Comparar ventanas para ambas condiciones
windows = [2, 4, 6, 8, 10]  # ms

results_baseline_windows = {}
results_coupled_windows = {}

for window in windows:
    analyzer_w = ForwardPropagationAnalyzer(window_ms=window)
    
    res_b = analyzer_w.analyze_single_spikes(
        spike_dict_baseline, neighbors_baseline, min_spikes=5
    )
    res_c = analyzer_w.analyze_single_spikes(
        spike_dict_coupled, neighbors_coupled, min_spikes=5
    )
    
    results_baseline_windows[window] = res_b
    results_coupled_windows[window] = res_c
    
    print(f"Window {window}ms: σ_baseline={res_b['sigma']:.3f}, "
          f"σ_coupled={res_c['sigma']:.3f}")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sigma_b = [results_baseline_windows[w]['sigma'] for w in windows]
sigma_c = [results_coupled_windows[w]['sigma'] for w in windows]
ax.plot(windows, sigma_b, 'o--', linewidth=2, markersize=8, 
        color='gray', label='Baseline (K=0)')
ax.plot(windows, sigma_c, 'o-', linewidth=2, markersize=8, 
        color='coral', label=f'Coupled (K={k_factor})')
ax.axhline(1.0, color='green', linestyle=':', alpha=0.5, label='Critical')
ax.set_xlabel('Window (ms)', fontsize=11)
ax.set_ylabel('σ (branching)', fontsize=11)
ax.set_title('Branching Ratio vs Window', fontsize=12)
ax.grid(alpha=0.3)
ax.legend()

ax = axes[1]
P_b = [results_baseline_windows[w]['P_transmission_global'] for w in windows]
P_c = [results_coupled_windows[w]['P_transmission_global'] for w in windows]
ax.plot(windows, P_b, 'o--', linewidth=2, markersize=8, 
        color='gray', label='Baseline')
ax.plot(windows, P_c, 'o-', linewidth=2, markersize=8, 
        color='steelblue', label='Coupled')
ax.set_xlabel('Window (ms)', fontsize=11)
ax.set_ylabel('P_transmission', fontsize=11)
ax.set_title('Transmission Probability vs Window', fontsize=12)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Setup
import os
import sys
from pathlib import Path

if Path.cwd().name == 'two_populations':
    os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt
from brian2 import *
from datetime import datetime
from collections import defaultdict

from src.two_populations.model import IzhikevichNetwork
from src.two_populations.metrics import analyze_simulation_results, print_network_statistics_table
from src.two_populations.plots.basic_plots import plot_raster_results
from src.two_populations.plots.dashboard_plots import plot_population_dashboard, plot_connectivity_dashboard
from src.two_populations.helpers.logger import setup_logger
from src.two_populations.helpers.validator import (
    add_validation_to_analysis, 
    plot_population_validation_dashboard, 
    print_validation_summary
)
from src.two_populations.metrics import analyze_simulation_results
from src.two_populations.plots.basic_plots import plot_raster_results, plot_voltage_traces

logger = setup_logger(
    experiment_name="spike_propagation",
    console_level="INFO",
    file_level="DEBUG",
    log_to_file=False
)

logger.info(f"Working directory: {Path.cwd()}")


# ANALYSIS WITH VALIDATION
# =============================================================================

def results_summary(results, network):
    # Analyze connectivity
    results_dict = {
        'baseline': analyze_simulation_results(
        results['A']['spike_monitor'], 
        None, 
        1000, "Baseline", 
        warmup=SIM_CONFIG['warmup_ms'],
        state_monitors={'A': network.monitors['A']},
        signal_mode = 'lfp',
        T_total=SIM_CONFIG['T_ms'] # firing_rate
        )
    }
    

    # Add validation metrics
    validation_results = add_validation_to_analysis(results_dict)

    # Network statistics table
    print_network_statistics_table(
        results, network, 
        N_exc=Ne, N_inh=Ni, 
        T_total=SIM_CONFIG['T_ms'], 
        warmup=500 #SIM_CONFIG['warmup_ms']
    )
    
    fig_conn = plot_raster_results(results)
    fig = plot_voltage_traces(results_dict, results, (500,1500), 3, 3, True, 3)
    
    # Dashboards
    fig_val = plot_population_validation_dashboard(validation_results)
    print_validation_summary(validation_results)

    fig_pop = plot_population_dashboard(results_dict)
    plt.show()
    
    
    
def simulate(k, SIM_CONFIG, BASE_PARAMS):
    
    # K=0: Solo input externo, sin acoplamiento recurrente
    start_scope()
    k_factor = k

    network_baseline = IzhikevichNetwork(
        dt_val=SIM_CONFIG['dt_ms'],
        T_total=SIM_CONFIG['T_ms'],
        fixed_seed=100,
        variable_seed=200,
        trial=0
    )

    params_baseline = {**BASE_PARAMS, 'k_exc': k_factor, 'k_inh': k_factor*3.9}
    pop_A_baseline = network_baseline.create_population2(name='A', **params_baseline)

    network_baseline.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
    results_baseline = network_baseline.run_simulation()

    logger.success(f"Baseline K=0: {len(results_baseline['A']['spike_times'])} spikes")
    
from matplotlib.colors import TwoSlopeNorm

# ============================================================================
# BARRIDO 2D: K × WINDOW
# ============================================================================

# Parámetros de simulación
Ni = 200
Ne = 800
time_ms = 5000

SIM_CONFIG = {
    'dt_ms': 0.1,
    'T_ms': time_ms,
    'warmup_ms': 500
}

# Parámetros base (estímulo constante)
BASE_PARAMS = {
    'Ne': Ne, 'Ni': Ni,
    'noise_exc': 0.884, 
    'noise_inh': 0.60,
    'p_intra': 0.1, 
    'delay': 0.0,
    'rate_hz': 8.0,
    'stim_start_ms': None, 
    'stim_duration_ms': time_ms,  # Estímulo constante
    'stim_base': 1.0,          # Sin modulación
    'stim_elevated': None
}

logger.info(f"Configuración: Ne={Ne}, Ni={Ni}, T={time_ms}ms (estímulo constante)")
    

class ForwardPropagationAnalyzer:
    """
    Analiza propagación forward: cuando neurona i dispara, ¿cuántos vecinos j responden?
    
    Métricas:
        - P_transmission: probabilidad de activación por spike
        - σ (sigma): branching ratio = <n_activados>
    """
    
    def __init__(self, window_ms=5.0):
        self.window = window_ms
    
    def extract_connectivity_E2E(self, synapses_intra, Ne, min_weight):
        """Extrae conectividad estructural E→E"""
        neighbors = defaultdict(list)
        weights = {}
        
        pre_indices = np.array(synapses_intra.i)
        post_indices = np.array(synapses_intra.j)
        syn_weights = np.array(synapses_intra.w)
        
        # Stats pre-filtro
        total_synapses = len(pre_indices)
        E2E_mask = (pre_indices < Ne) & (post_indices < Ne) & (syn_weights >= 0.0)
        n_E2E = np.sum(E2E_mask)
        
        # Filtro por peso
        mask = E2E_mask & (syn_weights >= min_weight)
        n_filtered = np.sum(mask)
        
        logger.info(f"  Total synapses: {total_synapses}")
        logger.info(f"  E→E (w>0): {n_E2E}")
        logger.info(f"  E→E (w>{min_weight}): {n_filtered} ({100*n_filtered/n_E2E:.1f}%)")
        if n_filtered > 0:
            logger.info(f"  Weight range: [{syn_weights[mask].min():.3f}, {syn_weights[mask].max():.3f}]")
        else:
            logger.warning(f"  No connections pass min_weight threshold!")
        
        for pre, post, w in zip(pre_indices[mask], post_indices[mask], syn_weights[mask]):
            neighbors[int(pre)].append(int(post))
            weights[(int(pre), int(post))] = float(w)
        
        # Out-degree stats
        degrees = [len(v) for v in neighbors.values()]
        if degrees:
            logger.info(f"  Out-degree: mean={np.mean(degrees):.1f}, median={np.median(degrees):.0f}, max={np.max(degrees)}")
        
        return dict(neighbors), weights
        
    def organize_spike_times(self, spike_times_arr, spike_indices_arr):
        """Organiza spikes por neurona"""
        spike_dict = defaultdict(list)
        
        for t, idx in zip(spike_times_arr, spike_indices_arr):
            spike_dict[int(idx)].append(float(t))
        
        spike_dict = {k: np.sort(v) for k, v in spike_dict.items()}
        return dict(spike_dict)
    
    def count_responses_single_spike(self, pre_spike_time, post_neuron_spikes):
        """Verifica si post respondió en ventana temporal"""
        if len(post_neuron_spikes) == 0:
            return False
        
        responses = post_neuron_spikes[(post_neuron_spikes > pre_spike_time) &  #TODO DEFINIRMOS > O >=??
                                    (post_neuron_spikes < pre_spike_time + self.window)]
        return len(responses) > 0
    
    def analyze_single_spikes(self, spike_dict, neighbors, min_spikes=5):
        """Análisis principal con logging"""
        logger.info(f"  Neurons with spikes: {len(spike_dict)}")
        logger.info(f"  Neurons with neighbors: {len(neighbors)}")
        logger.info(f"  Min spikes threshold: {min_spikes}")
        
        ratios_per_spike = []
        activated_counts = []
        per_neuron_stats = {}
        
        total_spikes_analyzed = 0
        successful_transmissions = 0
        neurons_excluded_min_spikes = 0
        
        for pre_idx in neighbors.keys():
            if pre_idx not in spike_dict:
                continue
            
            pre_spikes = spike_dict[pre_idx]
            
            if len(pre_spikes) < min_spikes:
                neurons_excluded_min_spikes += 1
                continue
            
            post_neighbors = neighbors[pre_idx]
            n_neighbors = len(post_neighbors)
            
            if n_neighbors == 0:
                continue
            
            neuron_ratios = []
            neuron_activated = []
            
            for spike_time in pre_spikes:
                n_activated = 0
                
                for post_idx in post_neighbors:
                    if post_idx not in spike_dict:
                        continue
                    
                    post_spikes = spike_dict[post_idx]
                    
                    if self.count_responses_single_spike(spike_time, post_spikes):
                        n_activated += 1
                
                ratio = n_activated / n_neighbors
                
                ratios_per_spike.append(ratio)
                activated_counts.append(n_activated)
                neuron_ratios.append(ratio)
                neuron_activated.append(n_activated)
                
                total_spikes_analyzed += 1
                if n_activated > 0:
                    successful_transmissions += 1
            
            per_neuron_stats[pre_idx] = {
                'n_spikes': len(pre_spikes),
                'n_neighbors': n_neighbors,
                'mean_ratio': np.mean(neuron_ratios),
                'std_ratio': np.std(neuron_ratios),
                'mean_activated': np.mean(neuron_activated),
                'success_rate': np.sum(np.array(neuron_activated) > 0) / len(neuron_activated)
            }
            
        logger.info(f"  Neurons excluded (min_spikes): {neurons_excluded_min_spikes}")
        logger.info(f"  Neurons analyzed: {len(per_neuron_stats)}")
        logger.info(f"  Total spikes analyzed: {total_spikes_analyzed}")
        
        ratios_per_spike = np.array(ratios_per_spike)
        activated_counts = np.array(activated_counts)
        
        results = {
            'P_transmission_global': np.mean(ratios_per_spike) if len(ratios_per_spike) > 0 else 0.0,
            'sigma': np.mean(activated_counts) if len(activated_counts) > 0 else 0.0,
            'ratio_distribution': ratios_per_spike,
            'activated_counts': activated_counts,
            'stats': {
                'n_total_spikes': total_spikes_analyzed,
                'n_successful': successful_transmissions,
                'n_failed': total_spikes_analyzed - successful_transmissions,
                'success_rate': successful_transmissions / total_spikes_analyzed if total_spikes_analyzed > 0 else 0.0,
                'n_neurons_analyzed': len(per_neuron_stats),
                'std_ratio': np.std(ratios_per_spike) if len(ratios_per_spike) > 0 else 0.0,
                'std_sigma': np.std(activated_counts) if len(activated_counts) > 0 else 0.0
            },
            'per_neuron': per_neuron_stats,
            'window_ms': self.window
        }
        
        return results

logger.info("ForwardPropagationAnalyzer cargado")

In [ ]:
from matplotlib.colors import TwoSlopeNorm

# ============================================================================
# BARRIDO 2D: K × WINDOW
# ============================================================================

# Parámetros de simulación
Ni = 200
Ne = 800
time_ms = 5000

SIM_CONFIG = {
    'dt_ms': 0.1,
    'T_ms': time_ms,
    'warmup_ms': 500
}

# Parámetros base (estímulo constante)
BASE_PARAMS = {
    'Ne': Ne, 'Ni': Ni,
    'noise_exc': 0.884, 
    'noise_inh': 0.60,
    'p_intra': 0.1, 
    'delay': 0.0,
    'rate_hz': 8.0,
    'stim_start_ms': None, 
    'stim_duration_ms': time_ms,  # Estímulo constante
    'stim_base': 1.0,          # Sin modulación
    'stim_elevated': None
}

logger.info(f"Configuración: Ne={Ne}, Ni={Ni}, T={time_ms}ms (estímulo constante)")

# =============================================================================

# Parámetros del barrido
K_values = np.arange(0.75, 6, 1) 
window_values = np.arange(1, 20, 2)  # ms
min_spikes = 10

# Inicializar estructuras para resultados
n_K = len(K_values)
n_win = len(window_values)
results = {
    'sigma_baseline': np.zeros((n_K, n_win)),
    'sigma_coupled': np.zeros((n_K, n_win)),
    'delta_sigma': np.zeros((n_K, n_win)),
    'N_baseline': np.zeros((n_K, n_win)),
    'N_coupled': np.zeros((n_K, n_win)),
    'P_trans_baseline': np.zeros((n_K, n_win)),
    'P_trans_coupled': np.zeros((n_K, n_win))
}

print("="*80)
print("BARRIDO 2D: K × WINDOW")
print("="*80)
print(f"K range: {K_values}")
print(f"Window range: {window_values} ms")
print(f"Total simulations: {n_K * n_win * 2} (baseline + coupled)")
print(f"Estimated time: ~{n_K * n_win * 2 * 0.5:.1f} min")
print("="*80)

start_total = time.time()

for i, K in enumerate(K_values):
    for j, window_ms in enumerate(window_values):
        print(f"\n[{i*n_win + j + 1}/{n_K*n_win}] K={K:.1f}, window={window_ms}ms")
        
        # ========== BASELINE (K=0.5 siempre, min_weight=0) ==========
        print("  Running BASELINE...")
        start_scope()
        k_factor = K

        network_baseline = IzhikevichNetwork(
            dt_val=SIM_CONFIG['dt_ms'],
            T_total=SIM_CONFIG['T_ms'],
            fixed_seed=100,
            variable_seed=200,
            trial=0
        )
        
        params_baseline = {**BASE_PARAMS, 'k_exc': 0.25, 'k_inh': 0.975} 
        pop_A_baseline = network_baseline.create_population2(name='A', **params_baseline)

        network_baseline.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
        results_baseline = network_baseline.run_simulation()

        analyzer = ForwardPropagationAnalyzer(window_ms=window_ms)
        
        # === BASELINE (K=0) ===
        logger.info("Analizando baseline (K=0)...")
        spike_dict_baseline = analyzer.organize_spike_times(
            results_baseline['A']['spike_times'], 
            results_baseline['A']['spike_indices']
        )
        neighbors_baseline, _ = analyzer.extract_connectivity_E2E(
            network_baseline.populations['A']['syn_intra'], Ne, 0.0
        )
        res_base = analyzer.analyze_single_spikes(
            spike_dict_baseline, neighbors_baseline, min_spikes=10
        )
        
        # ========== COUPLED (K variable, min_weight=0.5) ==========
        print("  Running COUPLED...")
        start_scope()
        
        network_coupled = IzhikevichNetwork(
            dt_val=SIM_CONFIG['dt_ms'],
            T_total=SIM_CONFIG['T_ms'],
            fixed_seed=100,
            variable_seed=200,
            trial=0
        )
        
        params_coupled = {**BASE_PARAMS, 'k_exc': k_factor, 'k_inh': k_factor*3.9}
        pop_A_coupled = network_coupled.create_population2(name='A', **params_coupled)

        network_coupled.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
        results_coupled = network_coupled.run_simulation()
 
        # === BASELINE (K=0) ===
        logger.info(f"Analizando coupled (K={K})...")
        spike_dict_coupled = analyzer.organize_spike_times(
            results_coupled['A']['spike_times'], 
            results_coupled['A']['spike_indices']
        )
        neighbors_coupled, _ = analyzer.extract_connectivity_E2E(
            network_coupled.populations['A']['syn_intra'], Ne, 0.0
        )
        res_coup = analyzer.analyze_single_spikes(
            spike_dict_coupled, neighbors_coupled, min_spikes=10
        )
        
        # Guardar resultados
        results['sigma_baseline'][i, j] = res_base['sigma']
        results['sigma_coupled'][i, j] = res_coup['sigma']
        results['delta_sigma'][i, j] = res_coup['sigma'] - res_base['sigma']
        results['N_baseline'][i, j] = res_base['stats']['n_neurons_analyzed']
        results['N_coupled'][i, j] = res_coup['stats']['n_neurons_analyzed']
        results['P_trans_baseline'][i, j] = res_base['P_transmission_global']
        results['P_trans_coupled'][i, j] = res_coup['P_transmission_global']
        
        print(f"  σ: {res_base['sigma']:.3f} → {res_coup['sigma']:.3f} (Δ={results['delta_sigma'][i, j]:.3f})")

elapsed = time.time() - start_total
print(f"\n{'='*80}")
print(f"COMPLETED in {elapsed/60:.1f} min")
print(f"{'='*80}")

# ============================================================================
# VISUALIZACIÓN
# ============================================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Normalización para divergente centrado en 0 (para Δσ)
vmax_delta = np.max(np.abs(results['delta_sigma']))
norm_delta = TwoSlopeNorm(vmin=-vmax_delta, vcenter=0, vmax=vmax_delta)

# Panel 1: σ baseline
im1 = axes[0,0].imshow(results['sigma_baseline'], aspect='auto', cmap='Blues',
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[0,0].set_xlabel('Window (ms)')
axes[0,0].set_ylabel('K (baseline)')
axes[0,0].set_title('σ Baseline')
axes[0,0].set_yticks(K_values)
axes[0,0].set_xticks(window_values)
plt.colorbar(im1, ax=axes[0,0])

# Panel 2: σ coupled
im2 = axes[0,1].imshow(results['sigma_coupled'], aspect='auto', cmap='Oranges',
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[0,1].set_xlabel('Window (ms)')
axes[0,1].set_ylabel('K (coupled)')
axes[0,1].set_title('σ Coupled')
axes[0,1].set_yticks(K_values)
axes[0,1].set_xticks(window_values)
plt.colorbar(im2, ax=axes[0,1])

# Panel 3: Δσ (CRÍTICO)
im3 = axes[0,2].imshow(results['delta_sigma'], aspect='auto', cmap='RdBu_r',
                       norm=norm_delta,
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[0,2].set_xlabel('Window (ms)')
axes[0,2].set_ylabel('K')
axes[0,2].set_title('Δσ (Coupled - Baseline)')
axes[0,2].set_yticks(K_values)
axes[0,2].set_xticks(window_values)
axes[0,2].contour(window_values, K_values, results['delta_sigma'], 
                  levels=[0], colors='black', linewidths=2, linestyles='--')
plt.colorbar(im3, ax=axes[0,2])

# Panel 4: N neurons baseline
im4 = axes[1,0].imshow(results['N_baseline'], aspect='auto', cmap='Greens',
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[1,0].set_xlabel('Window (ms)')
axes[1,0].set_ylabel('K')
axes[1,0].set_title('N neurons Baseline')
axes[1,0].set_yticks(K_values)
axes[1,0].set_xticks(window_values)
plt.colorbar(im4, ax=axes[1,0])

# Panel 5: N neurons coupled
im5 = axes[1,1].imshow(results['N_coupled'], aspect='auto', cmap='Greens',
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[1,1].set_xlabel('Window (ms)')
axes[1,1].set_ylabel('K')
axes[1,1].set_title('N neurons Coupled')
axes[1,1].set_yticks(K_values)
axes[1,1].set_xticks(window_values)
plt.colorbar(im5, ax=axes[1,1])

# Panel 6: Ratio N_coupled/N_baseline
ratio_N = results['N_coupled'] / (results['N_baseline'] + 1e-6)
im6 = axes[1,2].imshow(ratio_N, aspect='auto', cmap='plasma',
                       extent=[window_values[0]-1, window_values[-1]+1,
                               K_values[0]-0.25, K_values[-1]+0.25], origin='lower')
axes[1,2].set_xlabel('Window (ms)')
axes[1,2].set_ylabel('K')
axes[1,2].set_title('N ratio (Coupled/Baseline)')
axes[1,2].set_yticks(K_values)
axes[1,2].set_xticks(window_values)
plt.colorbar(im6, ax=axes[1,2])

plt.tight_layout()
plt.show()

# ============================================================================
# TABLA RESUMEN
# ============================================================================
print("\n" + "="*100)
print("TABLA RESUMEN: Δσ(K, window)")
print("="*100)
header = "K \\ window |" + "".join([f"{w:>8}ms |" for w in window_values])
print(header)
print("-" * len(header))
for i, K in enumerate(K_values):
    row = f"   {K:<5.1f}   |"
    for j in range(n_win):
        row += f"  {results['delta_sigma'][i,j]:>6.3f} |"
    print(row)
print("="*100)

In [ ]:
# ============================================================================
# COMPARACIÓN BASELINE vs COUPLED (1 trial)
# ============================================================================

# Parámetros
Ne, Ni = 800, 200
SIM_CONFIG = {'dt_ms': 0.1, 'T_ms': 5000, 'warmup_ms': 500}
BASE_PARAMS = {
    'Ne': Ne, 'Ni': Ni,
    'noise_exc': 0.884, 
    'noise_inh': 0.60,
    'p_intra': 0.1, 
    'delay': 0.0,
    'rate_hz': 8.0,
    'stim_start_ms': None, 
    'stim_duration_ms': 5000,
    'stim_base': 1.0,
    'stim_elevated': None
}

# Análisis
window_ms = 5.0
min_spikes = 10

# ========== BASELINE (K=0.25) ==========
start_scope()
net_base = IzhikevichNetwork(dt_val=0.1, T_total=5000, fixed_seed=100, variable_seed=200, trial=0)
net_base.create_population2(name='A', **{**BASE_PARAMS, 'k_exc': 0.25, 'k_inh': 0.975})
net_base.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
res_base = net_base.run_simulation()

analyzer = ForwardPropagationAnalyzer(window_ms=window_ms)
spike_dict_base = analyzer.organize_spike_times(res_base['A']['spike_times'], res_base['A']['spike_indices'])
neighbors_base, _ = analyzer.extract_connectivity_E2E(net_base.populations['A']['syn_intra'], Ne, 0.0)
metrics_base = analyzer.analyze_single_spikes(spike_dict_base, neighbors_base, min_spikes=min_spikes)

# ========== COUPLED (K=2.5) ==========
start_scope()
net_coup = IzhikevichNetwork(dt_val=0.1, T_total=5000, fixed_seed=100, variable_seed=200, trial=0)
net_coup.create_population2(name='A', **{**BASE_PARAMS, 'k_exc': 2.5, 'k_inh': 9.75})
net_coup.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
res_coup = net_coup.run_simulation()

spike_dict_coup = analyzer.organize_spike_times(res_coup['A']['spike_times'], res_coup['A']['spike_indices'])
neighbors_coup, _ = analyzer.extract_connectivity_E2E(net_coup.populations['A']['syn_intra'], Ne, 0.0)
metrics_coup = analyzer.analyze_single_spikes(spike_dict_coup, neighbors_coup, min_spikes=min_spikes)

# ========== RESUMEN ==========
print("="*80)
print("COMPARACIÓN BASELINE vs COUPLED")
print("="*80)
print(f"{'Métrica':<25} {'Baseline (K=0.25)':<20} {'Coupled (K=2.5)':<20} {'Δ':<15}")
print("-"*80)
print(f"{'σ (branching ratio)':<25} {metrics_base['sigma']:<20.3f} {metrics_coup['sigma']:<20.3f} {metrics_coup['sigma']-metrics_base['sigma']:<15.3f}")
print(f"{'P_transmission':<25} {metrics_base['P_transmission_global']:<20.4f} {metrics_coup['P_transmission_global']:<20.4f} {metrics_coup['P_transmission_global']-metrics_base['P_transmission_global']:<15.4f}")
print(f"{'Success rate':<25} {metrics_base['stats']['success_rate']:<20.3f} {metrics_coup['stats']['success_rate']:<20.3f} {metrics_coup['stats']['success_rate']-metrics_base['stats']['success_rate']:<15.3f}")
print(f"{'N neurons analyzed':<25} {metrics_base['stats']['n_neurons_analyzed']:<20} {metrics_coup['stats']['n_neurons_analyzed']:<20} {metrics_coup['stats']['n_neurons_analyzed']-metrics_base['stats']['n_neurons_analyzed']:<15}")
print(f"{'N spikes analyzed':<25} {metrics_base['stats']['n_total_spikes']:<20} {metrics_coup['stats']['n_total_spikes']:<20} {metrics_coup['stats']['n_total_spikes']-metrics_base['stats']['n_total_spikes']:<15}")
print("="*80)

# ========== VISUALIZACIÓN ==========
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Raster plots
for ax, res, title in zip(axes[0,:2], [res_base, res_coup], ['Baseline (K=0.25)', 'Coupled (K=2.5)']):
    ax.scatter(res['A']['spike_times'], res['A']['spike_indices'], s=0.5, c='k', alpha=0.5)
    ax.set_xlim(0, 1000)
    ax.set_ylim(0, 1000)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Neuron index')
    ax.set_title(title)

# σ distributions
axes[0,2].hist(metrics_base['activated_counts'], bins=30, alpha=0.5, label='Baseline', density=True)
axes[0,2].hist(metrics_coup['activated_counts'], bins=30, alpha=0.5, label='Coupled', density=True)
axes[0,2].axvline(metrics_base['sigma'], color='C0', linestyle='--', label=f'σ_base={metrics_base["sigma"]:.2f}')
axes[0,2].axvline(metrics_coup['sigma'], color='C1', linestyle='--', label=f'σ_coup={metrics_coup["sigma"]:.2f}')
axes[0,2].set_xlabel('N activated per spike')
axes[0,2].set_ylabel('Density')
axes[0,2].legend()
axes[0,2].set_title('Branching distributions')

# Firing rates
for ax, res, title in zip(axes[1,:2], [res_base, res_coup], ['Baseline', 'Coupled']):
    rates = []
    for i in range(Ne):
        n_spikes = np.sum(res['A']['spike_indices'] == i)
        rates.append(n_spikes / 5.0)
    ax.hist(rates, bins=30, color='gray', alpha=0.7)
    ax.axvline(np.mean(rates), color='r', linestyle='--', label=f'Mean={np.mean(rates):.2f} Hz')
    ax.set_xlabel('Firing rate (Hz)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.set_title(f'{title} - Firing rates')

# P_transmission comparison
ax = axes[1,2]
metrics_list = [metrics_base, metrics_coup]
labels = ['Baseline', 'Coupled']
colors = ['C0', 'C1']
for i, (m, l, c) in enumerate(zip(metrics_list, labels, colors)):
    ax.bar(i, m['P_transmission_global'], color=c, alpha=0.7, label=l)
ax.set_xticks([0,1])
ax.set_xticklabels(labels)
ax.set_ylabel('P_transmission')
ax.set_title('Transmission probability')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# ANÁLISIS ROBUSTO: Propagación de spikes con trial-shuffled baseline
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from brian2 import start_scope
from collections import defaultdict
import multiprocessing as mp
from functools import partial
import time

from src.two_populations.model import IzhikevichNetwork
from src.two_populations.helpers.logger import setup_logger

logger = setup_logger("spike_propagation", console_level="INFO", file_level="DEBUG", log_to_file=False)

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
Ne, Ni, time_ms = 800, 200, 10000

SIM_CONFIG = {
    'dt_ms': 0.1,
    'T_ms': time_ms,
    'warmup_ms': 500
}

BASE_PARAMS = {
    'Ne': Ne, 'Ni': Ni,
    'noise_exc': 0.884, 
    'noise_inh': 0.60,
    'p_intra': 0.1, 
    'delay': 0.0,
    'rate_hz': 8.0,
    'stim_start_ms': None, 
    'stim_duration_ms': time_ms,
    'stim_base': 1.0,
    'stim_elevated': None
}

# ============================================================================
# ANALIZADOR BASE
# ============================================================================
class ForwardPropagationAnalyzer:
    def __init__(self, window_ms=5.0):
        self.window = window_ms
    
    def extract_connectivity_E2E(self, synapses_intra, Ne, min_weight):
        neighbors = defaultdict(list)
        weights = {}
        
        pre_indices = np.array(synapses_intra.i)
        post_indices = np.array(synapses_intra.j)
        syn_weights = np.array(synapses_intra.w)
        
        E2E_mask = (pre_indices < Ne) & (post_indices < Ne) & (syn_weights >= 0.0)
        mask = E2E_mask & (syn_weights >= min_weight)
        
        for pre, post, w in zip(pre_indices[mask], post_indices[mask], syn_weights[mask]):
            neighbors[int(pre)].append(int(post))
            weights[(int(pre), int(post))] = float(w)
        
        return dict(neighbors), weights
        
    def organize_spike_times(self, spike_times_arr, spike_indices_arr):
        spike_dict = defaultdict(list)
        for t, idx in zip(spike_times_arr, spike_indices_arr):
            spike_dict[int(idx)].append(float(t))
        return {k: np.sort(v) for k, v in spike_dict.items()}
    
    def count_responses_single_spike(self, pre_spike_time, post_neuron_spikes):
        if len(post_neuron_spikes) == 0:
            return False
        responses = post_neuron_spikes[(post_neuron_spikes > pre_spike_time) & 
                                      (post_neuron_spikes < pre_spike_time + self.window)]
        return len(responses) > 0
    
    def analyze_single_spikes(self, spike_dict, neighbors, min_spikes=5):
        ratios_per_spike = []
        activated_counts = []
        
        for pre_idx in neighbors.keys():
            if pre_idx not in spike_dict or len(spike_dict[pre_idx]) < min_spikes:
                continue
            
            pre_spikes = spike_dict[pre_idx]
            post_neighbors = neighbors[pre_idx]
            n_neighbors = len(post_neighbors)
            
            if n_neighbors == 0:
                continue
            
            for spike_time in pre_spikes:
                n_activated = 0
                for post_idx in post_neighbors:
                    if post_idx in spike_dict:
                        if self.count_responses_single_spike(spike_time, spike_dict[post_idx]):
                            n_activated += 1
                
                ratios_per_spike.append(n_activated / n_neighbors)
                activated_counts.append(n_activated)
        
        ratios_per_spike = np.array(ratios_per_spike)
        activated_counts = np.array(activated_counts)
        
        return {
            'P_transmission_global': np.mean(ratios_per_spike) if len(ratios_per_spike) > 0 else 0.0,
            'sigma': np.mean(activated_counts) if len(activated_counts) > 0 else 0.0,
            'stats': {
                'n_total_spikes': len(ratios_per_spike),
                'n_neurons_analyzed': len([k for k in neighbors.keys() if k in spike_dict and len(spike_dict[k]) >= min_spikes])
            }
        }

# ============================================================================
# ANALIZADOR ROBUSTO
# ============================================================================
class RobustPropagationAnalyzer(ForwardPropagationAnalyzer):
    def shuffle_test_global(self, spike_dict, neighbors, n_shuffles=150, min_spikes=25):
        """Shuffle global: redistribuye spikes entre neuronas"""
        shuffled_sigmas = []
        
        # Extraer todos spikes
        all_neurons, all_times = [], []
        for nid, times in spike_dict.items():
            all_neurons.extend([nid] * len(times))
            all_times.extend(times)
        
        for i in range(n_shuffles):
            np.random.seed(i + 1000)
            # Permutar qué neurona dispara cada spike
            shuffled_neurons = np.random.permutation(all_neurons)
            
            shuffled_dict = defaultdict(list)
            for nid, t in zip(shuffled_neurons, all_times):
                shuffled_dict[nid].append(t)
            shuffled_dict = {k: np.sort(v) for k, v in shuffled_dict.items()}
            
            res = self.analyze_single_spikes(shuffled_dict, neighbors, min_spikes)
            shuffled_sigmas.append(res['sigma'])
        
        return np.array(shuffled_sigmas)
    
    def weight_dependence(self, spike_dict, neighbors, weights, n_bins=10):
        weight_list, p_trans_list = [], []
        
        for pre_idx in neighbors.keys():
            if pre_idx not in spike_dict or len(spike_dict[pre_idx]) < 5:
                continue
            
            pre_times = spike_dict[pre_idx]
            for post_idx in neighbors[pre_idx]:
                if post_idx not in spike_dict:
                    continue
                
                w = weights.get((pre_idx, post_idx), 0)
                if w == 0:
                    continue
                
                post_times = spike_dict[post_idx]
                n_activated = sum([self.count_responses_single_spike(t, post_times) for t in pre_times])
                p_trans = n_activated / len(pre_times)
                
                weight_list.append(w)
                p_trans_list.append(p_trans)
        
        if len(weight_list) == 0:
            return {'weight_bins': [], 'p_trans_mean': [], 'p_trans_std': [], 'counts': []}
        
        weight_arr = np.array(weight_list)
        p_trans_arr = np.array(p_trans_list)
        
        w_min, w_max = weight_arr.min(), weight_arr.max()
        bin_edges = np.linspace(w_min, w_max, n_bins + 1)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        
        bin_means, bin_stds, bin_counts = [], [], []
        for i in range(n_bins):
            mask = (weight_arr >= bin_edges[i]) & (weight_arr < bin_edges[i+1])
            if i == n_bins - 1:
                mask = (weight_arr >= bin_edges[i]) & (weight_arr <= bin_edges[i+1])
            
            vals = p_trans_arr[mask]
            if len(vals) > 0:
                bin_means.append(np.mean(vals))
                bin_stds.append(np.std(vals))
                bin_counts.append(len(vals))
            else:
                bin_means.append(np.nan)
                bin_stds.append(np.nan)
                bin_counts.append(0)
        
        return {
            'weight_bins': bin_centers,
            'p_trans_mean': np.array(bin_means),
            'p_trans_std': np.array(bin_stds),
            'counts': np.array(bin_counts)
        }

# ============================================================================
# MULTIPROCESSING
# ============================================================================
def run_single_coupled_trial_mp(trial, sim_config, base_params):
    start_scope()
    net = IzhikevichNetwork(dt_val=sim_config['dt_ms'], T_total=sim_config['T_ms'],
                           fixed_seed=100, variable_seed=200+trial, trial=trial)
    
    params = {**base_params, 'k_exc': 2.5, 'k_inh': 9.75}
    net.create_population2(name='A', **params)
    net.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
    res = net.run_simulation()
    
    analyzer = RobustPropagationAnalyzer(window_ms=5.0)
    spike_dict = analyzer.organize_spike_times(res['A']['spike_times'], res['A']['spike_indices'])
    neighbors, weights = analyzer.extract_connectivity_E2E(net.populations['A']['syn_intra'], 800, 0.5)
    metrics = analyzer.analyze_single_spikes(spike_dict, neighbors, min_spikes=25)
    
    return {'trial': trial, 'metrics': metrics, 'spike_dict': spike_dict, 
            'neighbors': neighbors, 'weights': weights}

def run_trial_shuffled_analysis_mp(n_trials=30, K_baseline=0.25):
    # BASELINE
    logger.info("Running baseline...")
    start_scope()
    net_base = IzhikevichNetwork(dt_val=0.1, T_total=10000, fixed_seed=100, variable_seed=200, trial=0)
    net_base.create_population2(name='A', **{**BASE_PARAMS, 'k_exc': K_baseline, 'k_inh': K_baseline*3.9})
    net_base.setup_monitors(['A'], record_v_dt=0.5, sample_fraction=0.5)
    res_base = net_base.run_simulation()
    
    analyzer = RobustPropagationAnalyzer(window_ms=5.0)
    spike_dict_base = analyzer.organize_spike_times(res_base['A']['spike_times'], res_base['A']['spike_indices'])
    neighbors_base, _ = analyzer.extract_connectivity_E2E(net_base.populations['A']['syn_intra'], 800, 0.0)
    metrics_base = analyzer.analyze_single_spikes(spike_dict_base, neighbors_base, min_spikes=25)
    
    # COUPLED (parallel)
    trials = list(range(1, n_trials+1))
    n_processes = min(mp.cpu_count()-1, n_trials)
    worker_func = partial(run_single_coupled_trial_mp, sim_config=SIM_CONFIG, base_params=BASE_PARAMS)
    
    logger.info(f"Running {n_trials} coupled trials ({n_processes} processes)...")
    with mp.Pool(processes=n_processes) as pool:
        coupled_results = pool.map(worker_func, trials)
        
    # Después de pool.map, verificar:
    print("\nDEBUG: Individual trial results")
    for r in coupled_results:
        print(f"Trial {r['trial']}: σ={r['metrics']['sigma']:.4f}, "
            f"n_spikes={r['metrics']['stats']['n_total_spikes']}")
    
    results_coupled = [r['metrics'] for r in coupled_results]
    results_network = [{'delta_sigma': r['metrics']['sigma'] - metrics_base['sigma'],
                        'delta_p_trans': r['metrics']['P_transmission_global'] - metrics_base['P_transmission_global']}
                       for r in coupled_results]
    
    return {
        'baseline': metrics_base, 
        'baseline_neighbors': neighbors_base,  # ← AÑADIR
        'coupled': results_coupled, 
        'network_only': results_network, 
        'coupled_full': coupled_results
    }

# ============================================================================
# EJECUTAR
# ============================================================================
start_time = time.time()
results = run_trial_shuffled_analysis_mp(n_trials=30, K_baseline=0.25)

# Resumen
coupled_sigmas = [r['sigma'] for r in results['coupled']]
delta_sigmas = [r['delta_sigma'] for r in results['network_only']]

print("\n" + "="*80)
print("TRIAL-SHUFFLED ANALYSIS")
print("="*80)
print(f"Time: {(time.time()-start_time)/60:.1f} min")
print(f"BASELINE: σ={results['baseline']['sigma']:.3f}, P_trans={results['baseline']['P_transmission_global']:.4f}")
print(f"COUPLED:  σ={np.mean(coupled_sigmas):.3f}±{np.std(coupled_sigmas):.3f}")
print(f"NETWORK:  Δσ={np.mean(delta_sigmas):.3f}±{np.std(delta_sigmas):.3f}")
print("="*80)

# Shuffle test
trial1 = results['coupled_full'][0]
shuffled = RobustPropagationAnalyzer(window_ms=5.0).shuffle_test_global(
    trial1['spike_dict'], trial1['neighbors'], n_shuffles=150, min_spikes=15)
z_score = (trial1['metrics']['sigma'] - np.mean(shuffled)) / np.std(shuffled)
print(f"\nShuffle: Real={trial1['metrics']['sigma']:.3f}, Null={np.mean(shuffled):.3f}±{np.std(shuffled):.3f}, Z={z_score:.2f}")

# Weight dependence
weight_dep = RobustPropagationAnalyzer(window_ms=5.0).weight_dependence(
    trial1['spike_dict'], trial1['neighbors'], trial1['weights'], n_bins=15)

fig, ax = plt.subplots(figsize=(8, 5))
valid = ~np.isnan(weight_dep['p_trans_mean'])
ax.errorbar(weight_dep['weight_bins'][valid], weight_dep['p_trans_mean'][valid], 
            yerr=weight_dep['p_trans_std'][valid], fmt='o-', capsize=5)
ax.set_xlabel('Weight')
ax.set_ylabel('P_transmission')
ax.set_title('P_trans vs weight')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
results

In [ ]:
# ============================================================================
# ANÁLISIS Y VISUALIZACIÓN EXTENDIDA
# ============================================================================

# Después del resumen básico, añadir:

# 1. VALIDACIÓN: Estadísticas de conectividad
print("\n" + "="*80)
print("CONNECTIVITY STATS")
print("="*80)
trial1 = results['coupled_full'][0]

neighbors_b = results['baseline_neighbors']  # ← Usar esto
out_deg_base = [len(v) for v in neighbors_b.values()]
out_deg_coupled = [len(v) for v in trial1['neighbors'].values()]

print(f"BASELINE (min_w=0.0): mean_deg={np.mean(out_deg_base):.1f}, median={np.median(out_deg_base):.0f}")
print(f"COUPLED (min_w=0.5):  mean_deg={np.mean(out_deg_coupled):.1f}, median={np.median(out_deg_coupled):.0f}")
print(f"Weight range (coupled): [{min(trial1['weights'].values()):.2f}, {max(trial1['weights'].values()):.2f}]")

# 2. TABLA COMPARATIVA
print("\n" + "="*80)
print("PROPAGATION METRICS COMPARISON")
print("="*80)
print(f"{'Metric':<25} {'Baseline':<15} {'Coupled':<15} {'Δ (net only)':<15}")
print("-"*80)
print(f"{'σ (branching)':<25} {results['baseline']['sigma']:<15.3f} {np.mean(coupled_sigmas):<15.3f} {np.mean(delta_sigmas):<15.3f}")
print(f"{'P_transmission':<25} {results['baseline']['P_transmission_global']:<15.4f} {np.mean([r['P_transmission_global'] for r in results['coupled']]):<15.4f} {np.mean([r['delta_p_trans'] for r in results['network_only']]):<15.4f}")
print(f"{'CV(σ)':<25} {'N/A':<15} {np.std(coupled_sigmas)/np.mean(coupled_sigmas):<15.3f} {'-':<15}")
print(f"{'N neurons analyzed':<25} {results['baseline']['stats']['n_neurons_analyzed']:<15} {np.mean([r['stats']['n_neurons_analyzed'] for r in results['coupled']]):<15.0f} {'-':<15}")

# 3. SHUFFLE TEST
shuffled = RobustPropagationAnalyzer(window_ms=5.0).shuffle_test(
    trial1['spike_dict'], trial1['neighbors'], n_shuffles=150, min_spikes=25)
z_score = (trial1['metrics']['sigma'] - np.mean(shuffled)) / np.std(shuffled)
print(f"\nSHUFFLE TEST: Real={trial1['metrics']['sigma']:.3f}, Null={np.mean(shuffled):.3f}±{np.std(shuffled):.3f}, Z={z_score:.2f}")
print(f"Significance: p < {1 - (z_score/3.09):.4f}" if z_score > 3.09 else "Not significant (p > 0.001)")

# 4. PLOTS
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Distribución σ
ax = axes[0, 0]
ax.hist([results['baseline']['sigma']], bins=1, alpha=0.7, label='Baseline', color='blue', range=(0, 4))
ax.hist(coupled_sigmas, bins=15, alpha=0.7, label='Coupled', color='orange')
ax.axvline(results['baseline']['sigma'], color='blue', linestyle='--', linewidth=2)
ax.axvline(np.mean(coupled_sigmas), color='orange', linestyle='--', linewidth=2)
ax.set_xlabel('σ (branching ratio)')
ax.set_ylabel('Count')
ax.legend()
ax.set_title('Branching ratio distribution')
ax.grid(True, alpha=0.3)

# Plot 2: P_transmission
ax = axes[0, 1]
coupled_ptrans = [r['P_transmission_global'] for r in results['coupled']]
ax.hist([results['baseline']['P_transmission_global']], bins=1, alpha=0.7, label='Baseline', color='blue', range=(0, 0.1))
ax.hist(coupled_ptrans, bins=15, alpha=0.7, label='Coupled', color='orange')
ax.set_xlabel('P_transmission')
ax.set_ylabel('Count')
ax.legend()
ax.set_title('Transmission probability')
ax.grid(True, alpha=0.3)

# Plot 3: Weight dependence
ax = axes[0, 2]
weight_dep = RobustPropagationAnalyzer(window_ms=5.0).weight_dependence(
    trial1['spike_dict'], trial1['neighbors'], trial1['weights'], n_bins=15)
valid = ~np.isnan(weight_dep['p_trans_mean'])
ax.errorbar(weight_dep['weight_bins'][valid], weight_dep['p_trans_mean'][valid], 
            yerr=weight_dep['p_trans_std'][valid], fmt='o-', capsize=5, color='green')
ax.set_xlabel('Synaptic weight')
ax.set_ylabel('P_transmission')
ax.set_title('P_trans vs weight')
ax.grid(True, alpha=0.3)

# Plot 4: Shuffle test
ax = axes[1, 0]
ax.hist(shuffled, bins=30, alpha=0.7, color='gray', label='Shuffled null')
ax.axvline(trial1['metrics']['sigma'], color='red', linestyle='--', linewidth=2, label=f'Real (Z={z_score:.1f})')
ax.set_xlabel('σ (shuffled)')
ax.set_ylabel('Count')
ax.legend()
ax.set_title('Shuffle test')
ax.grid(True, alpha=0.3)

# Plot 5: Trial variability
ax = axes[1, 1]
trial_nums = [r['trial'] for r in results['coupled_full']]
trial_sigmas = [r['metrics']['sigma'] for r in results['coupled_full']]
ax.scatter(trial_nums, trial_sigmas, alpha=0.6, s=50)
ax.axhline(np.mean(trial_sigmas), color='red', linestyle='--', label=f'Mean={np.mean(trial_sigmas):.2f}')
ax.fill_between([0, max(trial_nums)+1], 
                np.mean(trial_sigmas)-np.std(trial_sigmas), 
                np.mean(trial_sigmas)+np.std(trial_sigmas), 
                alpha=0.2, color='red')
ax.set_xlabel('Trial')
ax.set_ylabel('σ')
ax.legend()
ax.set_title('Inter-trial variability')
ax.grid(True, alpha=0.3)

# Plot 6: Δσ distribution
ax = axes[1, 2]
ax.hist(delta_sigmas, bins=15, alpha=0.7, color='purple')
ax.axvline(np.mean(delta_sigmas), color='red', linestyle='--', linewidth=2, 
          label=f'Mean={np.mean(delta_sigmas):.2f}±{np.std(delta_sigmas):.2f}')
ax.set_xlabel('Δσ (coupled - baseline)')
ax.set_ylabel('Count')
ax.legend()
ax.set_title('Network contribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. VALIDACIÓN ADICIONAL: Correlación weight-P_trans
print("\n" + "="*80)
print("WEIGHT-TRANSMISSION CORRELATION")
print("="*80)
valid_bins = ~np.isnan(weight_dep['p_trans_mean'])
if np.sum(valid_bins) > 2:
    corr = np.corrcoef(weight_dep['weight_bins'][valid_bins], 
                      weight_dep['p_trans_mean'][valid_bins])[0, 1]
    print(f"Pearson r = {corr:.3f}")
    print(f"Interpretation: {'Positive correlation ✓' if corr > 0.3 else 'Weak/no correlation ✗'}")

In [ ]:
# Añadir al notebook después de los resultados
trial1 = results['coupled_full'][0]
analyzer = RobustPropagationAnalyzer(window_ms=5.0)

# DEBUG SHUFFLE
print("\n" + "="*80)
print("SHUFFLE TEST DEBUG")
print("="*80)

# Original
sigma_original = trial1['metrics']['sigma']
print(f"σ original: {sigma_original:.4f}")

# Test 3 shuffles manualmente
shuffled_sigmas = []
for i in range(3):
    np.random.seed(i + 1000)
    shuffled_dict = {nid: np.random.permutation(times).copy() 
                    for nid, times in trial1['spike_dict'].items()}
    
    # Verificar que cambió
    neuron_test = list(trial1['spike_dict'].keys())[0]
    original_times = trial1['spike_dict'][neuron_test][:5]
    shuffled_times = shuffled_dict[neuron_test][:5]
    
    print(f"\nShuffle {i}:")
    print(f"  Neuron {neuron_test} original[:5]: {original_times}")
    print(f"  Neuron {neuron_test} shuffled[:5]: {shuffled_times}")
    print(f"  Times changed: {not np.allclose(original_times, shuffled_times)}")
    
    # Analizar
    res = analyzer.analyze_single_spikes(shuffled_dict, trial1['neighbors'], min_spikes=25)
    shuffled_sigmas.append(res['sigma'])
    print(f"  σ shuffled: {res['sigma']:.4f}")

print(f"\nσ shuffled: mean={np.mean(shuffled_sigmas):.4f}, std={np.std(shuffled_sigmas):.6f}")
print(f"Expected: std > 0 si shuffle efectivo")

# Verificar si permutation mantiene ordering
print("\n" + "="*80)
print("PERMUTATION TEST")
print("="*80)
test_times = np.array([1.5, 2.3, 5.1, 7.8, 10.2])
np.random.seed(1000)
perm1 = np.random.permutation(test_times)
np.random.seed(1000)
perm2 = np.random.permutation(test_times)
print(f"Original: {test_times}")
print(f"Perm1:    {perm1}")
print(f"Perm2:    {perm2}")
print(f"Identical: {np.allclose(perm1, perm2)}")

# Hipótesis: Permutar times ordenados no rompe estructura temporal lo suficiente
# porque count_responses_single_spike solo cuenta spikes en ventana [t, t+window)
# Si la distribución temporal se preserva localmente, σ no cambia

print("\n" + "="*80)
print("TEMPORAL STRUCTURE TEST")
print("="*80)
# Comparar ISI distributions
neuron_sample = list(trial1['spike_dict'].keys())[10]
original = trial1['spike_dict'][neuron_sample]
np.random.seed(1000)
shuffled = np.random.permutation(original)

if len(original) > 1:
    isi_orig = np.diff(np.sort(original))
    isi_shuf = np.diff(np.sort(shuffled))
    print(f"Neuron {neuron_sample}:")
    print(f"  ISI original: mean={np.mean(isi_orig):.2f}, std={np.std(isi_orig):.2f}")
    print(f"  ISI shuffled: mean={np.mean(isi_shuf):.2f}, std={np.std(isi_shuf):.2f}")
    print(f"  Rate preserved: {np.isclose(np.mean(isi_orig), np.mean(isi_shuf))}")